# Semantic Link Labs: The Inspection
### Best Practice Analyzer
> *"Something's bothering me. I don't know what it is yet. But something's not right."* — Adrian Monk
---
Monk never leaves a scene until everything checks out.
The **Best Practice Analyzer (BPA)** is SLL's version of Monk walking your semantic model
and flagging everything that doesn't sit right — naming conventions, unused columns,
missing descriptions, formatting violations, DAX anti-patterns.
By default, the BPA checks more than 60 rules against your semantic model.
The rules come from experts within Microsoft and the Fabric Community, and violations
are organized into five categories: **Performance, DAX Expressions, Error Prevention, Maintenance, and Formatting.**
He will find something. He always does.
| Step | Theme | What We Do |
|---|---|---|
| 1 | The Walk-Through | Run BPA on a single model with default rules |
| 2 | Filing the Report | Persist results to a delta table |
| 3 | Clearing the District | Scan every model in the workspace |
| 4 | The Operations Center | Build a Direct Lake model and report on BPA results |

---
## Case File #0 — Stocking the Evidence Room
This notebook keeps its setup here so the investigation can use the configuration below
without calling a separate setup notebook.
**Before the demo:** prepare Semantic Link Labs **0.18.1** in this notebook's
session or attached Fabric Environment. Run the optional install cell only if needed.
**Changing targets:** edit the model names, report (where applicable), and workspace
in the configuration cell. A different schema may also require changes in the investigation cells.
**Runtime:** Attach EvidenceLocker for the BPA export, bulk scan, and reporting sections.

### Inline setup used in this demo
Run the configuration and imports below. The installation cell is optional preparation.
If installation requests a session restart, restart before running the configuration and imports.

In [ ]:
# Optional preparation: run this cell only if the pinned package is not already installed.
%pip install semantic-link-labs==0.18.1 --quiet

In [ ]:
# Configuration: change the investigation targets here.
WORKSPACE = None  # None = the current Fabric workspace
DATASET_BPA = "SFCaseFiles"
DATASET_BPA_AUTO_DATE = "Trudoso10K_WithAutoDate"

In [ ]:
import importlib.metadata
import notebookutils
import sempy.fabric as fabric
import sempy_labs as labs
import pandas as pd
import sempy_labs.report as rep

sll_version = importlib.metadata.version("semantic-link-labs")
print(f"Semantic Link Labs: {sll_version} (demo target: 0.18.1)")

context = notebookutils.runtime.context
WORKSPACE_ID = WORKSPACE or context["currentWorkspaceId"]
WORKSPACE_NAME = context["currentWorkspaceName"]
LAKEHOUSE_ID = context.get("defaultLakehouseId")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

print(f"Notebook workspace: {WORKSPACE_NAME}")
print(f"Model workspace: {WORKSPACE or WORKSPACE_NAME}")
print(f"Main model: {DATASET_BPA}")
print(f"Lakehouse for BPA exports: {LAKEHOUSE_NAME or 'No default Lakehouse attached'}")

---
## Step 1 — "The Walk-Through"

> *Monk walks the scene methodically. Every room. Every corner. He doesn't skip anything.*

Run the BPA against a single model with default rules.
Results display inline in the notebook — violations grouped by category and severity.

`extended=True` runs `set_vertipaq_annotations()` first, which feeds real VertiPaq statistics
into the rule engine before evaluation. Rules that check column cardinality, table size,
and memory footprint will use actual numbers rather than estimates.
On small models the output looks identical — the difference shows up on large production models.

In [ ]:
# Run BPA with default rules — 60+ rules, inline display
# Tip: extended=True runs set_vertipaq_annotations() first for more accurate
# size/cardinality rules on large production models (return_dataframe=True to get the raw data)

labs.run_model_bpa(dataset=DATASET_BPA, workspace=WORKSPACE)

### The Rule Catalog
`model_bpa_rules()` returns the full set of rules BPA evaluates — name, category, severity, and description.
Worth displaying before you run so the audience can see what they're about to get.

In [ ]:
df = labs.model_bpa_rules()
display(df)

### Targeted Scans — Running a Rule Subset
You don't have to run all 60+ rules every time. Filter `model_bpa_rules()` to any subset
and pass it in via the `rules` parameter.
Here we pull two rules that apply directly to the auto-date model (`DATASET_BPA_AUTO_DATE`) —
one for the auto-date tables themselves, one for a common DAX filter pattern they introduce.

In [ ]:
rules = labs.model_bpa_rules()

__rule_subset = rules[
    (rules['Rule Name'] == "Remove auto-date table") |
    (rules['Rule Name'] == "Filter column values with proper syntax")
].copy()

display(__rule_subset)

Run BPA against the auto-date model using only the targeted rule subset.
Cleaner output, faster execution, directly relevant findings.

In [ ]:
labs.run_model_bpa(
    dataset=DATASET_BPA_AUTO_DATE,
    rules=__rule_subset
    )

---
## Step 2 — "Filing the Report"

> *Monk writes everything down. In the case file. In order. With timestamps.*

Setting `export=True` writes the BPA results to a **`modelbparesults` delta table**
in your attached lakehouse — so violations are persisted, queryable, and trackable over time.

This is where BPA becomes a **governance process** rather than a one-time check.
Run it on a schedule, append results, trend your violations over sprints.

In [ ]:
# Export results to delta table in the attached lakehouse
# Appends to 'modelbparesults' — safe to run repeatedly
labs.run_model_bpa(dataset=DATASET_BPA, workspace=WORKSPACE, export=True)

print(f"BPA results exported to '{LAKEHOUSE_NAME}/modelbparesults'")

### Bonus: Internationalization
Pass a `language` parameter and BPA translates rule names, categories, and descriptions dynamically.
The violations are the same — the output language changes.
Useful for teams working across regions.

In [ ]:
# Bonus: run in a different language
# Rules, categories and descriptions are dynamically translated
labs.run_model_bpa(dataset=DATASET_BPA, workspace=WORKSPACE, language="French")

---
## Step 3 — "Clearing the District"

> *One case solved. But Monk notices the whole district has the same problem.*

`run_model_bpa_bulk` scans **every semantic model in a workspace** in one call,
appending all results to the same `modelbparesults` delta table.

This is the enterprise play — automated BPA across every model your team owns.
No manual runs. No missed models. Monk inspects the whole building, not just one apartment.

In [ ]:
# Scan every semantic model in the current workspace
# Appends all results to 'modelbparesults' delta table

labs.run_model_bpa_bulk(workspace=WORKSPACE_ID)

print("✅ Bulk BPA complete — all models in workspace scanned")

---
## Step 4 — "The Operations Center"
> *Every good detective has a wall. Monk has a very organized wall.*
SLL can build a full **Direct Lake semantic model and Power BI report** on top of your
`modelbparesults` delta table — two function calls.
- `create_model_bpa_semantic_model()` — creates the **BPAModel** semantic model in Direct Lake mode
- `create_model_bpa_report()` — creates the **ModelBPA** Power BI report bound to that model
Since it's Direct Lake, as you append new BPA runs the report updates automatically —
no model refresh required. Run `run_model_bpa_bulk()` on a schedule and the wall updates itself.
*Monk approves.*

In [ ]:
# Step 1 — Create the BPA semantic model (Direct Lake on modelbparesults)
labs.create_model_bpa_semantic_model()

print("BPAModel semantic model created in Direct Lake mode")

In [ ]:
# Step 2 — Create the BPA Power BI report
rep.create_model_bpa_report()

print("BPAReport created")
print(f"Navigate to your workspace to open the report and view results")
print("Going forward just run 'run_model_bpa_bulk' — results appear automatically")

### Launch the Report
`launch_report()` opens the ModelBPA report directly from the notebook.
The report is already populated — Step 3 appended results to `modelbparesults`
and the Direct Lake model picked them up automatically.

In [ ]:
rep.launch_report(report="ModelBPA")

---
## Ongoing Workflow

Once the model and report are built, the ongoing workflow is just one cell:

```python
# Run this on a schedule — weekly, per sprint, pre-release
labs.run_model_bpa_bulk(workspace=WORKSPACE)
```

Results append to `modelbparesults`, the Direct Lake model picks them up automatically,
and the report is always current. No refresh. No manual steps.

**Put it in a Fabric pipeline and you have automated model governance.**

*Monk would run it every day. Probably twice.*

---
## Wrap-Up

| Step | Function | What it does |
|---|---|---|
| 1 | `run_model_bpa()` | Single model — 60+ rules, inline display |
| 1 | `model_bpa_rules()` | Inspect and filter the rule catalog |
| 2 | `run_model_bpa(export=True)` | Persist results to `modelbparesults` delta table |
| 3 | `run_model_bpa_bulk()` | Scan every model in the workspace |
| 4 | `create_model_bpa_semantic_model()` | Build Direct Lake model on BPA results |
| 4 | `create_model_bpa_report()` | Build Power BI report on BPA model |
| 4 | `launch_report()` | Open the report directly from the notebook |

### Resources:
- [SLL GitHub — BPA Report notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Best%20Practice%20Analyzer%20Report.ipynb)
- [SLL GitHub — Model Optimization notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Model%20Optimization.ipynb)

> *"You know what I hate? Unformatted measures.*
> *You know what else I hate? Everything else.*" — Adrian Monk (probably)